# GEPA prompt optimization for campaign relevance

Short notebook to optimize the system prompt for the Langfuse dataset `campaign_relevance_disagree_subset_9d488308aa46` using the GEPA DefaultAdapter.

In [1]:
%pip install -q "gepa[full]"


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from typing import Dict, Any, Tuple, List
import dotenv

dotenv.load_dotenv()

from langfuse import Langfuse

DEFAULT_LANGFUSE_DATASET = "campaign_relevance_disagree_subset_9d488308aa46"


def load_langfuse_campaign(
    dataset_name: str = DEFAULT_LANGFUSE_DATASET,
) -> Tuple[Dict[str, bool], Dict[str, Any]]:
    dataset = Langfuse().get_dataset(dataset_name)

    truth: Dict[str, bool] = {}
    inputs: Dict[str, Any] = {}

    for item in dataset.items:
        cid = str((item.metadata or {}).get("custom_id") or item.id)
        if not cid:
            continue

        if item.expected_output and isinstance(item.expected_output, dict):
            truth[cid] = bool(item.expected_output.get("campaign_relevant", False))
        else:
            truth[cid] = False

        if item.input is not None:
            inputs[cid] = item.input

    return truth, inputs


def _to_multimodal_input(raw_input: Any) -> Dict[str, Any]:
    """
    Convert Langfuse/OpenAI batch-style input into structured multimodal data.

    Output shape:
    {
        "text_parts": [...],
        "image_urls": [...],
        "raw_user_messages": [...]
    }

    Notes:
    - Keeps only user messages.
    - Preserves text blocks separately from image URLs.
    - Drops system prompt content because GEPA optimizes that separately.
    """
    text_parts: List[str] = []
    image_urls: List[str] = []
    raw_user_messages: List[Dict[str, Any]] = []

    if isinstance(raw_input, list):
        for msg in raw_input:
            if not isinstance(msg, dict):
                continue
            if msg.get("role") != "user":
                continue

            raw_user_messages.append(msg)
            content = msg.get("content")

            # Plain text user message
            if isinstance(content, str):
                txt = content.strip()
                if txt:
                    text_parts.append(txt)
                continue

            # Multimodal content blocks
            if isinstance(content, list):
                for block in content:
                    if not isinstance(block, dict):
                        continue

                    block_type = block.get("type")

                    if block_type == "input_text":
                        txt = str(block.get("text") or "").strip()
                        if txt:
                            text_parts.append(txt)

                    elif block_type == "input_image":
                        image_url = block.get("image_url")
                        if image_url:
                            image_urls.append(str(image_url))

        return {
            "text_parts": text_parts,
            "image_urls": image_urls,
            "raw_user_messages": raw_user_messages,
        }

    # Fallback for unexpected shapes
    fallback_text = json.dumps(raw_input, ensure_ascii=False)
    return {
        "text_parts": [fallback_text],
        "image_urls": [],
        "raw_user_messages": [],
    }


def _label_to_gold(label: bool) -> str:
    return "TRUE" if label else "FALSE"


truth, inputs = load_langfuse_campaign()

trainset = []
for cid, inp in inputs.items():
    if cid not in truth:
        continue

    label = truth[cid]
    mm_input = _to_multimodal_input(inp)

    trainset.append(
        {
            # structured multimodal input for a custom GEPA adapter
            "input": mm_input,
            # binary gold label
            "answer": _label_to_gold(label),
            # useful metadata for debugging/evaluator feedback
            "additional_context": {
                "custom_id": cid,
                "num_text_parts": len(mm_input["text_parts"]),
                "num_images": len(mm_input["image_urls"]),
            },
        }
    )

print("trainset size:", len(trainset))
print("example input:", json.dumps(trainset[0]["input"], ensure_ascii=False, indent=2)[:2000])

trainset size: 300
example input: {
  "text_parts": [
    "Caption: So many great brands on the @douglas_cosmetics Beauty Day 🩵\n\n#douglas #beauty #makeup #vlog #düsseldorf #maccosmetics #soldejaneiro",
    "Transcription:",
    "Hashtags: ['douglas', 'beauty', 'makeup', 'vlog', 'düsseldorf', 'maccosmetics', 'soldejaneiro']",
    "Mentions: ['douglas_cosmetics']",
    "You are shown equally spaced frames extracted from the video",
    "Here is the preview image for the video:"
  ],
  "image_urls": [
    "https://d3lbkpzyekd5xi.cloudfront.net/ig/post/daeffcae-c750-460a-aa58-55b5bab3cd29/frames/1_full.jpg",
    "https://d3lbkpzyekd5xi.cloudfront.net/ig/post/daeffcae-c750-460a-aa58-55b5bab3cd29/frames/2_full.jpg",
    "https://d3lbkpzyekd5xi.cloudfront.net/ig/post/daeffcae-c750-460a-aa58-55b5bab3cd29/frames/3_full.jpg",
    "https://d3lbkpzyekd5xi.cloudfront.net/ig/post/daeffcae-c750-460a-aa58-55b5bab3cd29/frames/4_full.jpg",
    "https://d3lbkpzyekd5xi.cloudfront.net/ig/post/daeffcae-c7

In [3]:
import gepa
import json
from pathlib import Path

from ingest import ROOT

# Seed system prompt from the batch file used to create the dataset
batch_path = Path(ROOT) / "batches" / "openai_gpt-5-nano.jsonl"
with open(batch_path, "r", encoding="utf-8") as f:
    first_line = json.loads(next(f))

_seed_system_prompt = ""
for msg in first_line.get("body", {}).get("input", []):
    if msg.get("role") == "system":
        _seed_system_prompt = str(msg.get("content", "")).strip()
        break

# If somehow there is no system prompt, fall back to an empty one
seed_prompt = {"system_prompt": _seed_system_prompt}

from random import shuffle, seed as _seed

_seed(42)
shuffle(trainset)
split_idx = int(0.8 * len(trainset))
valset = trainset[split_idx:]
trainset = trainset[:split_idx]

len(trainset), len(valset)



(240, 60)

In [4]:
seed_prompt

{'system_prompt': '# Campaign Relevancy Assessment\n\n    ## Task Overview\n    You are analyzing content created by influencers to determine if it\'s relevant to a specific influencer marketing campaign. Your goal is to classify each content piece as either relevant or not relevant to the campaign.\n\n    ## Detailed Instructions\n    1. **Content Analysis:**\n       - Examine all provided content including captions, images, videos, and audio transcriptions\n       - Look for both direct mentions and contextual references to the campaign or brand\n\n    2. **Classification Criteria:** \n       - **Campaign-Relevant (TRUE)** if the content:\n         - Explicitly mentions one of the brands: "Douglas", "Estée Lauder", "La Mer", "Clinique", "Dr Jart+", "MAC", "Bobbi Brown", "Tom Ford", "Hugo Boss", "Yves Saint Laurent", "Prada", "Dolce & Gabbana", "Douglas Beauty Card", "Annemarie Börlind", "Sun Matters", "Gisada", "Lancôme", "Biotherm", "Annayake", "Typebea", "Drybar", "Morphe", "Gitti"

In [5]:
trainset

[{'input': {'text_parts': ['Transcription:',
    'Hashtags: []',
    "Mentions: ['barbaralovesbeauty', 'hudabeauty', 'douglas_cosmetics']",
    'You are shown equally spaced frames extracted from the video',
    'Here is the preview image for the video:'],
   'image_urls': ['https://d3lbkpzyekd5xi.cloudfront.net/ig/story/13426e6c-b144-4459-a97c-33316289107c/frames/1_full.jpg',
    'https://d3lbkpzyekd5xi.cloudfront.net/ig/story/13426e6c-b144-4459-a97c-33316289107c/frames/2_full.jpg',
    'https://d3lbkpzyekd5xi.cloudfront.net/ig/story/13426e6c-b144-4459-a97c-33316289107c/frames/3_full.jpg',
    'https://d3lbkpzyekd5xi.cloudfront.net/ig/story/13426e6c-b144-4459-a97c-33316289107c/frames/4_full.jpg',
    'https://d3lbkpzyekd5xi.cloudfront.net/ig/story/13426e6c-b144-4459-a97c-33316289107c/frames/5_full.jpg',
    'https://d3lbkpzyekd5xi.cloudfront.net/ig/story/13426e6c-b144-4459-a97c-33316289107c/frames/6_full.jpg',
    'https://d3lbkpzyekd5xi.cloudfront.net/ig/story/13426e6c-b144-4459-a97c

In [6]:
result = gepa.optimize(
    seed_candidate=seed_prompt,
    trainset=trainset,
    valset=valset,
    task_lm="openai/gpt-5-nano",
    reflection_lm="openai/gpt-5.1",
    max_metric_calls=500,
    display_progress_bar=True,
)

print("Best score:", result.val_aggregate_scores[result.best_idx])

GEPA Optimization:  12%|█▏        | 60/500 [01:39<12:06,  1.65s/rollouts]

Iteration 0: Base program full valset score: 0.65 over 60 / 60 examples
Iteration 1: Selected program 0 score: 0.65
Iteration 1: Proposed new text for system_prompt: You are an AI assistant that evaluates influencer content for relevance to a specific influencer marketing campaign and extracts structured metadata.

## 1. Task Overview

You receive, for each content piece (e.g., Instagram post, story, TikTok), a set of text parts (caption, transcription, hashtags, mentions) and potentially one or more images. Your job is to:

1. Decide if the content is relevant to the campaign.
2. Produce a rich, holistic description of the content.
3. Extract mentioned products.
4. Extract mentioned brands (from a fixed allowed list only).
5. Extract hashtags.
6. Explain your relevance decision.

Your output must strictly follow the required fields and constraints described below.

---

## 2. Campaign Context

- **Campaign Name:** `Douglas_Beauty Day 2025`
- **Campaign Description (paraphrased):**
  -

GEPA Optimization:  25%|██▌       | 126/500 [05:31<17:25,  2.79s/rollouts]

Iteration 1: Found a better program on the valset with score 0.85.
Iteration 1: Valset score for new program: 0.85 (coverage 60 / 60)
Iteration 1: Val aggregate for new program: 0.85
Iteration 1: Individual valset scores for new program: {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.0, 10: 1.0, 11: 1.0, 12: 1.0, 13: 1.0, 14: 1.0, 15: 1.0, 16: 1.0, 17: 1.0, 18: 1.0, 19: 1.0, 20: 1.0, 21: 1.0, 22: 0.0, 23: 1.0, 24: 1.0, 25: 1.0, 26: 1.0, 27: 1.0, 28: 1.0, 29: 1.0, 30: 1.0, 31: 0.0, 32: 0.0, 33: 1.0, 34: 0.0, 35: 1.0, 36: 1.0, 37: 1.0, 38: 1.0, 39: 1.0, 40: 1.0, 41: 1.0, 42: 0.0, 43: 1.0, 44: 1.0, 45: 0.0, 46: 0.0, 47: 0.0, 48: 1.0, 49: 1.0, 50: 1.0, 51: 1.0, 52: 1.0, 53: 1.0, 54: 1.0, 55: 1.0, 56: 1.0, 57: 1.0, 58: 0.0, 59: 1.0}
Iteration 1: New valset pareto front scores: {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.0, 10: 1.0, 11: 1.0, 12: 1.0, 13: 1.0, 14: 1.0, 15: 1.0, 16: 1.0, 17: 1.0, 18: 1.0, 19: 1.0, 20: 1.0, 21: 1.

GEPA Optimization:  26%|██▌       | 129/500 [05:48<17:58,  2.91s/rollouts]

...

Iteration 15: Valset score for new program: 0.8333333333333334 (coverage 60 / 60)
Iteration 15: Val aggregate for new program: 0.8333333333333334
Iteration 15: Individual valset scores for new program: {0: 1.0, 1: 1.0, 2: 0.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.0, 10: 1.0, 11: 1.0, 12: 0.0, 13: 1.0, 14: 1.0, 15: 0.0, 16: 1.0, 17: 1.0, 18: 1.0, 19: 1.0, 20: 1.0, 21: 1.0, 22: 1.0, 23: 1.0, 24: 1.0, 25: 1.0, 26: 1.0, 27: 1.0, 28: 1.0, 29: 1.0, 30: 1.0, 31: 0.0, 32: 1.0, 33: 1.0, 34: 0.0, 35: 1.0, 36: 1.0, 37: 1.0, 38: 1.0, 39: 1.0, 40: 1.0, 41: 1.0, 42: 1.0, 43: 0.0, 44: 1.0, 45: 1.0, 46: 1.0, 47: 0.0, 48: 1.0, 49: 1.0, 50: 1.0, 51: 1.0, 52: 1.0, 53: 1.0, 54: 1.0, 55: 0.0, 56: 1.0, 57: 0.0, 58: 0.0, 59: 1.0}
Iteration 15: New valset pareto front scores: {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.0, 10: 1.0, 11: 1.0, 12: 1.0, 13: 1.0, 14: 1.0, 15: 1.0, 16: 1.0, 17: 1.0, 18: 1.0, 19: 1.0, 20: 1.0, 21: 1.0, 22: 1.0, 23: 1.0, 24: 1.0, 25: 1

GEPA Optimization: 100%|█████████▉| 498/500 [33:32<00:09,  4.91s/rollouts]

Iteration 16: New subsample score 1.0 is not better than old score 2.0, skipping
Iteration 17: Selected program 3 score: 0.75
Iteration 17: Proposed new text for system_prompt: You are an AI assistant that evaluates influencer social content for relevancy to a specific Douglas influencer marketing campaign and extracts structured metadata.

Your behavior is strictly defined by this instruction. Do not rely on prior runs of this task; treat this as the ground truth specification.

1. TASK OVERVIEW

For EACH content piece (typically an Instagram Story, post, or Reel-style video) you receive:

- `text_parts`: structured text metadata, such as:
  - Caption
  - Transcription (may be empty)
  - Hashtags line (e.g., `Hashtags: ['tag1', 'tag2']`)
  - Mentions line (e.g., `Mentions: ['handle1', 'handle2']`)
  - Utility/meta text (e.g., "You are shown equally spaced frames extracted from the video", "Here is the preview image for the video", "Here is the image:")
- `image_urls`: one or more URLs

GEPA Optimization: 100%|█████████▉| 498/500 [36:27<00:08,  4.39s/rollouts]

Iteration 17: Valset score for new program: 0.7333333333333333 (coverage 60 / 60)
Iteration 17: Val aggregate for new program: 0.7333333333333333
Iteration 17: Individual valset scores for new program: {0: 1.0, 1: 1.0, 2: 0.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.0, 10: 0.0, 11: 0.0, 12: 1.0, 13: 1.0, 14: 1.0, 15: 1.0, 16: 1.0, 17: 1.0, 18: 1.0, 19: 1.0, 20: 1.0, 21: 1.0, 22: 0.0, 23: 1.0, 24: 1.0, 25: 1.0, 26: 1.0, 27: 1.0, 28: 1.0, 29: 1.0, 30: 1.0, 31: 0.0, 32: 0.0, 33: 1.0, 34: 0.0, 35: 0.0, 36: 1.0, 37: 1.0, 38: 1.0, 39: 1.0, 40: 1.0, 41: 1.0, 42: 0.0, 43: 0.0, 44: 1.0, 45: 0.0, 46: 1.0, 47: 0.0, 48: 1.0, 49: 1.0, 50: 1.0, 51: 0.0, 52: 1.0, 53: 1.0, 54: 1.0, 55: 0.0, 56: 1.0, 57: 0.0, 58: 0.0, 59: 1.0}
Iteration 17: New valset pareto front scores: {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.0, 10: 1.0, 11: 1.0, 12: 1.0, 13: 1.0, 14: 1.0, 15: 1.0, 16: 1.0, 17: 1.0, 18: 1.0, 19: 1.0, 20: 1.0, 21: 1.0, 22: 1.0, 23: 1.0, 24: 1.0, 25: 1

In [7]:
print("Initial system prompt (from batch):\n", seed_prompt["system_prompt"][:400], "...\n")
print("Best system prompt:\n", result.best_candidate["system_prompt"])
print("Best score:", result.val_aggregate_scores[result.best_idx])

Initial system prompt (from batch):
 # Campaign Relevancy Assessment

    ## Task Overview
    You are analyzing content created by influencers to determine if it's relevant to a specific influencer marketing campaign. Your goal is to classify each content piece as either relevant or not relevant to the campaign.

    ## Detailed Instructions
    1. **Content Analysis:**
       - Examine all provided content including captions, image ...

Best system prompt:
 You are an AI assistant that evaluates influencer content for relevance to a specific influencer marketing campaign and extracts structured metadata.

## 1. Task Overview

You receive, for each content piece (e.g., Instagram post, story, TikTok), a set of text parts (caption, transcription, hashtags, mentions) and potentially one or more images. Your job is to:

1. Decide if the content is relevant to the campaign.
2. Produce a rich, holistic description of the content.
3. Extract mentioned products.
4. Extract mentioned brands (f

In [8]:
# save new prompt
with open("new_prompt.txt", "w") as f:
    f.write(result.best_candidate["system_prompt"])

In [9]:
# old prompt
with open("old_prompt.txt", "w") as f:
    f.write(seed_prompt["system_prompt"])

In [9]:
# ok lets make the batch file for gpt nano 5 with new prompt instead of hte old one

